# Phase 1: Exploratory Data Analysis (EDA)
## Industrial Demand Forecasting and Sales Analytics System

This notebook performs the Exploratory Data Analysis for the alloy sales dataset to analyze historical trends, identify seasonality, address missing values/outliers, and answer key business questions:
1. Which alloy sells the most?
2. Which months have the highest demand?
3. Is demand seasonal?
4. Which regions contribute most sales?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import os

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

### 1. Load Data

In [ ]:
data_path = os.path.join("..", "data", "sales_data.csv")
df = pd.read_csv(data_path)
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
print(f"Dataset contains {len(df)} records.")
df.head()

### 2. General Data Audit & Cleaning
We check for missing values, duplicate records, outliers, and our date range.

In [ ]:
print("--- Missing Values Analysis ---")
print(df.isnull().sum())

print("\n--- Duplicate Rows ---")
print(f"Number of duplicates: {df.duplicated().sum()}")

print("\n--- Date Range ---")
print(f"Start Date: {df['Order_Date'].min()}")
print(f"End Date: {df['Order_Date'].max()}")

print("\n--- Quantity and Revenue Outliers (Description) ---")
df[['Quantity', 'Revenue']].describe()

### 3. Business Questions Analysis

#### Q1: Which alloy sells the most?

In [ ]:
alloy_sales = df.groupby('Alloy_Type').agg({'Quantity': 'sum', 'Revenue': 'sum'}).reset_index()
alloy_sales = alloy_sales.sort_values(by='Quantity', ascending=False)

print("--- Alloy Sales Summary ---")
print(alloy_sales.to_string(index=False))

# Plot Alloy-wise Sales
fig = px.bar(alloy_sales, x='Alloy_Type', y='Quantity', title='Total Demand (Quantity) by Alloy Type',
             color='Quantity', color_continuous_scale='Viridis')
fig.show()

#### Q2 & Q3: Which months have highest demand? Is demand seasonal?

In [ ]:
# Extract month and aggregate
df['Month_Num'] = df['Order_Date'].dt.month
df['Month_Name'] = df['Order_Date'].dt.strftime('%B')

monthly_sales = df.groupby(['Month_Num', 'Month_Name']).agg({'Quantity': 'sum', 'Revenue': 'sum'}).reset_index()

fig = px.line(monthly_sales, x='Month_Name', y='Quantity', markers=True,
              title='Monthly Demand Seasonality Pattern',
              labels={'Quantity': 'Quantity Sold', 'Month_Name': 'Month'})
fig.show()

#### Q4: Which regions contribute most sales?

In [ ]:
region_sales = df.groupby('Region').agg({'Quantity': 'sum', 'Revenue': 'sum'}).reset_index()
region_sales = region_sales.sort_values(by='Revenue', ascending=False)

print("--- Regional Sales Summary ---")
print(region_sales.to_string(index=False))

# Plot Region Performance
fig = px.pie(region_sales, values='Revenue', names='Region', title='Sales Revenue Distribution by Region',
             hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

### 4. Over Time Demand & Revenue Trends
We plot the monthly aggregated demand over the entire timeline.

In [ ]:
df_monthly_trend = df.groupby(df['Order_Date'].dt.to_period('M')).agg({'Quantity': 'sum', 'Revenue': 'sum'}).reset_index()
df_monthly_trend['Order_Date'] = df_monthly_trend['Order_Date'].dt.to_timestamp()

# Plotting
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_monthly_trend['Order_Date'], y=df_monthly_trend['Quantity'],
                    mode='lines+markers', name='Quantity Sold', line=dict(color='royalblue', width=3)))

fig.update_layout(title='Monthly Demand Over Time (Quantity Trend)',
                  xaxis_title='Date', yaxis_title='Total Quantity')
fig.show()

fig_rev = go.Figure()
fig_rev.add_trace(go.Scatter(x=df_monthly_trend['Order_Date'], y=df_monthly_trend['Revenue'],
                    mode='lines+markers', name='Revenue ($)', line=dict(color='forestgreen', width=3)))
fig_rev.update_layout(title='Monthly Sales Revenue Over Time',
                  xaxis_title='Date', yaxis_title='Revenue ($)')
fig_rev.show()